In [ ]:
import os
os.chdir('???')
os.getcwd()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import datetime

### Load Dataset

In [ ]:
orig_df = pd.read_csv("shampoo_sales.csv")
orig_df.head(5)

In [ ]:
orig_df.shape

In [ ]:
orig_df.dtypes

In [ ]:
df = orig_df.copy()

### Set time index

#### String Time Format Code List: https://strftime.org/

In [ ]:
from datetime import datetime

def parser(x):
 return datetime.strptime('190'+x, '%Y-%m')

df['Month'] = df['Month'].apply(parser)
df.head()

In [ ]:
df.dtypes

In [ ]:
df = df.set_index('Month')
df.head()

In [ ]:
df.columns

In [ ]:
value_var = 'Sales'

In [ ]:
df[value_var].plot(title="Monthly Shampoo Sales", figsize=(8,4))
plt.show()

### Data Processing

In [ ]:
# Detecting and removing outliers using Z-score
from scipy.stats import zscore
z_scores = zscore(df[value_var])
abs_z_scores = np.abs(z_scores)

zscore_threshold = 3 #any data above mean +/- (this_threshold)*sd is considered outliers
outliers = (abs_z_scores > zscore_threshold)  # Z-score threshold.
df = df.loc[~outliers].copy()

print(orig_df.shape)
print(df.shape)

df.head()

In [ ]:
forecast_steps = 6  # Forecast duration

# because dataframe has only one column.  need to explicitly convert to dataframe
train_df = df[value_var][:-forecast_steps ].to_frame(name=value_var)
test_df  = df[value_var][-forecast_steps :].to_frame(name=value_var)
train_df

### Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
decomposition = seasonal_decompose(train_df[value_var], model='additive')  
fig = plt.figure(figsize=(20,8))  
decomposition.plot()  
plt.show()

In [ ]:
fig = plt.figure(figsize=(15,8))  
decomposition.seasonal.plot()
plt.show()

In [ ]:
# From the plot above, we can see that seasonal pattern lasts for 12 months (1 year)
# Set selected_s for ARIMA model
selected_s = ???

### Find differencing d 

In [ ]:
from statsmodels.tsa.stattools import adfuller
result = adfuller(train_df[value_var])
result

In [ ]:
def test_adfuller(series, alpha=0.05):
    from statsmodels.tsa.stattools import adfuller
    result = adfuller(series)
    print("ADF Statistic:", result[0])
    print("p-value:", result[1])
    for key, value in result[4].items():
        print(f"Critical Value ({key}): {value}")
    if result[1] <= alpha:
        print("Reject the null hypothesis. Data is stationary")
    else:
        print("Do not reject the null hypothesis. Data is not stationary ")

In [ ]:
test_adfuller(train_df[value_var])

In [ ]:
train_df['first_difference'] = (train_df[value_var] - train_df[value_var].shift(1)).dropna()
# or you can use df[value_var].diff().dropna()  # diff() = compute difference between t and t-1
train_df['first_difference'].head()

In [ ]:
train_df['first_difference'].plot(title="First-order Differenced Series", figsize=(8,4))
plt.show()

In [ ]:
test_adfuller(train_df['first_difference'].dropna())

In [ ]:
# Not needed in this example but to show how to find second difference
train_df['second_difference'] = train_df['first_difference'] - train_df['first_difference'].shift(1)
train_df['second_difference'].head()

In [ ]:
# Set selected_d for ARIMA/SARIMA model
selected_d = ???

### Find differencing D for seasonal timeseries 

In [ ]:
# Skip this section if timeseries is not seasonal

In [ ]:
train_df['seasonal_difference'] = train_df[value_var] - train_df[value_var].shift(selected_s)
train_df['seasonal_difference'].plot()
plt.show()

In [ ]:
test_adfuller(train_df['seasonal_difference'].dropna())

In [ ]:
train_df['seasonal_first_difference'] = train_df['first_difference'] - train_df['first_difference'].shift(selected_s)
train_df['seasonal_first_difference'].plot()

In [ ]:
test_adfuller(train_df['seasonal_first_difference'].dropna())

In [ ]:
# Set selected_D for SARIMA model
selected_D = ???

### Identify p and q from PACF and ACF

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf

fig, ax = plt.subplots(1, 2, figsize=(12,4))

plot_pacf(train_df['first_difference'].dropna(), lags=10, ax=ax[0])
plot_acf(train_df['first_difference'].dropna(), lags=10, ax=ax[1])

plt.show()

In [ ]:
# Set possible selected_p and selected_q for ARIMA/SARIMA model
selected_p = ???
selected_q = ???

### Identify P and Q from PACF and ACF for seasonal timeseries

In [ ]:
# Skip this section if timeseries is not seasonal

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf

fig, ax = plt.subplots(1, 2, figsize=(12,4))

plot_pacf(train_df['seasonal_first_difference'].dropna(), lags=6, ax=ax[0])
plot_acf(train_df['seasonal_first_difference'].dropna(), lags=6, ax=ax[1])

plt.show()

In [ ]:
# Set possible selected_P and selected_Q for SARIMA model
selected_P = ???
selected_Q = ???

### Select ARIMA model based on AIC and BIC

In [ ]:
# Set frequeuncy of time index 
# Frequenct String: https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases
train_df.index = pd.DatetimeIndex(train_df.index, freq='MS')
train_df.index.inferred_freq

In [ ]:
# Try to construct one model and read result
from statsmodels.tsa.arima.model import ARIMA

print('p: ' + str(selected_p))
print('d: ' + str(selected_d))
print('q: ' + str(selected_q))

model = ARIMA(train_df[value_var], order=(selected_p, selected_d, selected_q))
model_fit = model.fit()

print(model_fit.summary())

#===============================
# ar.L1: Strength of dependence on previous value
#        p-value < 0.05 -> AR term is statistically significant
#                        If insignificant → AR term may be unnecessary
# ma.L1: Captures shock/error correction
#        p-value < 0.05 → MA term contributes meaningfully
#                        If insignificant → MA term may be unnecessary#
# sigma² (noise variance): Variance of residuals
#        Smaller = model explains more structure
#===============================
# Ljung–Box: Test whether residuals are white noise
#            H0: residuals are uncorrelated
#            If p-value (Prob(Q)) > 0.05 -> residuals ≈ white noise -> ARIMA remove all temporal structure
# Heteroskedasticity (H): Test  whether variance of residuals is constant
#                         If p-value (Prob(H)) > 0.05 -> contant variance 
# Jarque-Bera (JB): Test whether residuals are normality
#                   If p-value (Prob(JB)) > 0.05 -> residuals roughly normal
#===============================

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))

residuals = model_fit.resid
residuals.plot(title="Residuals", ax=ax[0])
plot_acf(residuals, lags=10, ax=ax[1])
plt.show()

In [ ]:
# Construct multiple models 

In [ ]:
print(selected_p, selected_q)

In [ ]:
# Create all possible choices for p and q
start_p = 0
end_p = 2
start_q = 0
end_q = 2

pdq_choices = []
for pi in range(start_p, end_p+1):
    for qi in range(start_q, end_q+1):
        choice = (pi, selected_d, qi)
        pdq_choices.append(choice)
pdq_choices

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

for pdq_choice in pdq_choices:
    #try:
        print(pdq_choice)
        model = ARIMA(train_df[value_var], order=pdq_choice)
        model_fit = model.fit()
    
        # obtain prob(Q), prob(H), prob(JB)
        prob_Q = model_fit.test_serial_correlation(method='ljungbox')[0][1][0]
        prob_H = model_fit.test_heteroskedasticity(method='breakvar')[0][1]
        prob_JB = model_fit.test_normality(method='jarquebera')[0][1]

        print(pdq_choice, ", AIC:", model_fit.aic, ", Prob(Q):", prob_Q, ", Prob(H):", prob_H)
        print(model_fit.pvalues)
    #except:
    #    continue

#### ARIMA Forecast and Evaluation

In [ ]:
# Choose few choices of (p,q) to make predictions and evaluate results

In [ ]:
# Model 1
pdq_choice = (1,1,0)

model = ARIMA(train_df[value_var], order=pdq_choice)
model_fit = model.fit()
forecast = model_fit.forecast(steps=forecast_steps)

# aligne index
forecast.index = test_df.index

from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse = np.sqrt(mean_squared_error(test_df[value_var], forecast))
mae = mean_absolute_error(test_df[value_var], forecast)

print(pdq_choice, ", RMSE:", rmse, ", MAE :", mae)

In [ ]:
# Model 2
pdq_choice = (2,1,0)

model = ARIMA(train_df[value_var], order=pdq_choice)
model_fit = model.fit()
forecast = model_fit.forecast(steps=forecast_steps)

# aligne index
forecast.index = test_df.index

from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse = np.sqrt(mean_squared_error(test_df[value_var], forecast))
mae = mean_absolute_error(test_df[value_var], forecast)

print(pdq_choice, ", RMSE:", rmse, ", MAE :", mae)

In [ ]:
### Let's evaluate model (1,1,0)

In [ ]:
pdq_choice = (1,1,0)

model = ARIMA(train_df[value_var], order=pdq_choice)
model_fit = model.fit()
forecast = model_fit.forecast(steps=forecast_steps)
print(model_fit.summary())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))

residuals = model_fit.resid
residuals.plot(title="Residuals", ax=ax[0])
plot_acf(residuals, lags=10, ax=ax[1])
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(train_df[value_var], label="Train")
plt.plot(test_df[value_var], label="Test", marker="o")
plt.plot(test_df.index, forecast, label="Forecast", marker="x")
plt.legend()
plt.title("ARIMA Actual Values vs. Forecast")
plt.show()

### Select SARIMA model based on AIC and BIC

In [ ]:
# Set frequeuncy of time index 
# Frequenct String: https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases
train_df.index = pd.DatetimeIndex(train_df.index, freq='MS')
train_df.index.inferred_freq

In [ ]:
print(selected_s, selected_d, selected_D)
print(selected_p, selected_q)
print(selected_P, selected_Q)

In [ ]:
# Create all possible choices for p and q
start_p = 1
end_p = 1
start_q = 0
end_q = 1
start_P = 0
end_P = 2
start_Q = 0
end_Q = 2

pdq_choices = []
for pi in range(start_p, end_p+1):
    for qi in range(start_q, end_q+1):
        for Pi in range(start_P, end_P+1):
            for Qi in range(start_Q, end_Q+1):
                pdq_choice = (pi, selected_d, qi)
                PDQ_choice = (Pi, selected_D, Qi, selected_s)
                
                pdq_choices.append([pdq_choice, PDQ_choice])
print(pdq_choices)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.diagnostic import het_arch
from statsmodels.stats.stattools import jarque_bera

for pdq_choice in pdq_choices:
    #try:
        print(pdq_choice)
        model = SARIMAX(train_df[value_var],order=pdq_choice[0], seasonal_order=pdq_choice[1])
        model_fit = model.fit()
        # obtain prob(Q)
        lb = acorr_ljungbox(model_fit.resid, lags=[10], return_df=True)
        prob_Q = lb['lb_pvalue'].iloc[0]
        # obtain prob(H)
        prob_H = het_arch(model_fit.resid)[1]
        # obtain prob(JB)
        jb_stat, prob_JB, skew, kurt = jarque_bera(model_fit.resid)
        print(pdq_choice, ", AIC:", model_fit.aic, ", Prob(Q):", prob_Q, ", Prob(H):", prob_H)
        print(model_fit.pvalues)
    #except:
    #    continue

In [ ]:
# Choose few choices of (p,q) and P,D) to make predictions and evaluate results

In [ ]:
# Model 1
pdq_choice = [(1,1,0), (0,1,0,12)]

model = SARIMAX(train_df[value_var],order=pdq_choice[0], seasonal_order=pdq_choice[1])
model_fit = model.fit()
forecast = model_fit.forecast(steps=forecast_steps)

# aligne index
forecast.index = test_df.index

from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse = np.sqrt(mean_squared_error(test_df[value_var], forecast))
mae = mean_absolute_error(test_df[value_var], forecast)

print(pdq_choice, ", RMSE:", rmse, ", MAE :", mae)

fig, ax = plt.subplots(1, 2, figsize=(12,4))

ax[0].plot(train_df[value_var], label="Train")
ax[0].plot(test_df[value_var], label="Test", marker="o", color='green')
ax[0].plot(test_df.index, forecast, label="Forecast", marker="x", color='red')
ax[0].legend()
ax[0].set_title("Actual vs. Forecast")

ax[1].plot(test_df[value_var], label="Test", marker="o", color='green')
ax[1].plot(test_df.index, forecast, label="Forecast", marker="x", color='red')
ax[1].legend()
ax[1].set_title("Actual vs. Forecast")
ax[1].tick_params(axis='x', labelrotation=45)
ax[1].set_ylim([200, 700])
    
plt.show()

In [ ]:
# Model 2
pdq_choice = [(1,1,0), (1,1,0,12)]

model = SARIMAX(train_df[value_var],order=pdq_choice[0], seasonal_order=pdq_choice[1])
model_fit = model.fit()
forecast = model_fit.forecast(steps=forecast_steps)

# aligne index
forecast.index = test_df.index

from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse = np.sqrt(mean_squared_error(test_df[value_var], forecast))
mae = mean_absolute_error(test_df[value_var], forecast)

print(pdq_choice, ", RMSE:", rmse, ", MAE :", mae)

fig, ax = plt.subplots(1, 2, figsize=(12,4))

ax[0].plot(train_df[value_var], label="Train")
ax[0].plot(test_df[value_var], label="Test", marker="o", color='green')
ax[0].plot(test_df.index, forecast, label="Forecast", marker="x", color='red')
ax[0].legend()
ax[0].set_title("Actual vs. Forecast")

ax[1].plot(test_df[value_var], label="Test", marker="o", color='green')
ax[1].plot(test_df.index, forecast, label="Forecast", marker="x", color='red')
ax[1].legend()
ax[1].set_title("Actual vs. Forecast")
ax[1].tick_params(axis='x', labelrotation=45)
ax[1].set_ylim([200, 700])
    
plt.show()

In [ ]:
### Let's evaluate one model [(1, 1, 0), (1, 1, 0, 12)]

In [ ]:
pdq_choice = [(1, 1, 0), (1, 1, 0, 12)]

model = SARIMAX(train_df[value_var],order=pdq_choice[0], seasonal_order=pdq_choice[1])
model_fit = model.fit()
forecast = model_fit.forecast(steps=forecast_steps)
print(model_fit.summary())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))

residuals = model_fit.resid
residuals.plot(title="Residuals", ax=ax[0])
plot_acf(residuals, lags=10, ax=ax[1])
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(train_df[value_var], label="Train")
plt.plot(test_df[value_var], label="Test", marker="o")
plt.plot(test_df.index, forecast, label="Forecast", marker="x")
plt.legend()
plt.title("Actual vs. Forecast")
plt.show()

In [ ]:
# Plot with confidence intervals of forecasting values

plt.figure(figsize=(8,4))

# Confidence intervals
model_forecast = model_fit.get_forecast(steps=forecast_steps)
forecast_mean = model_forecast.predicted_mean                 # Predicted values, same result as model_fit.forecast(steps=forecast_steps)
forecast_ci = model_forecast.conf_int()

plt.plot(train_df[value_var], label="Train")
plt.plot(test_df[value_var], label="Test", marker="o")
plt.plot(forecast_mean, label="Forecast", marker="x")

# Plot confidence intervals
plt.fill_between(
    forecast_ci.index,
    forecast_ci.iloc[:, 0],
    forecast_ci.iloc[:, 1],
    alpha=0.3
)

plt.legend()
plt.title("Actual vs. Forecast")
plt.show()